In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')

In [ ]:
stock_files = glob.glob('../../data/raw/[A-Z]*.csv')
stock_list = []

for file in stock_files:
    if 'raw_analyst_ratings' in file:
        continue
    df = pd.read_csv(file)
    ticker = os.path.basename(file).replace('.csv', '')
    df['stock'] = ticker
    stock_list.append(df)

all_stocks_df = pd.concat(stock_list, ignore_index=True)
all_stocks_df['Date'] = pd.to_datetime(all_stocks_df['Date']).dt.normalize()

In [ ]:
news_df = pd.read_csv('../../data/raw/raw_analyst_ratings.csv')
news_df.dropna(subset=['headline', 'date', 'stock'], inplace=True)
news_df['date'] = pd.to_datetime(news_df['date'], errors='coerce', format='mixed', utc=True).dt.normalize()
news_df.dropna(subset=['date'], inplace=True)
news_df['date'] = news_df['date'].dt.tz_localize(None)

sia = SentimentIntensityAnalyzer()
news_df['sentiment_score'] = news_df['headline'].apply(lambda x: sia.polarity_scores(str(x))['compound'])
daily_sentiment = news_df.groupby(['date', 'stock'])['sentiment_score'].mean().reset_index()

In [ ]:
all_stocks_df.columns = all_stocks_df.columns.str.strip().str.lower()
daily_sentiment.columns = daily_sentiment.columns.str.strip().str.lower()
close_col = 'adj close' if 'adj close' in all_stocks_df.columns else 'close'

valid_tickers = all_stocks_df['stock'].unique()
daily_sentiment = daily_sentiment[daily_sentiment['stock'].isin(valid_tickers)].reset_index(drop=True)

all_stocks_df = all_stocks_df.sort_values(by=['stock', 'date']).reset_index(drop=True)
all_stocks_df['daily_return'] = all_stocks_df.groupby('stock')[close_col].pct_change() * 100
all_stocks_df.dropna(subset=['daily_return'], inplace=True)

daily_sentiment = daily_sentiment.sort_values(by='date')
all_stocks_df = all_stocks_df.sort_values(by='date')

merged_df = pd.merge_asof(daily_sentiment, all_stocks_df, on='date', by='stock', direction='forward')
merged_df.dropna(subset=['daily_return'], inplace=True)

In [ ]:
corr_value = merged_df['sentiment_score'].corr(merged_df['daily_return'], method='pearson')
print(f'Pearson Correlation Coefficient: {corr_value:.4f}')

plt.figure(figsize=(8, 5))
sns.scatterplot(data=merged_df, x='sentiment_score', y='daily_return', alpha=0.4, color='teal')
plt.title(f'News Sentiment vs Daily Stock Returns (Correlation: {corr_value:.4f})')
plt.xlabel('Average Daily Sentiment Score')
plt.ylabel('Daily Stock Return (%)')
plt.grid(True)
plt.show()

In [ ]:
def get_category(score):
    if score > 0.05: return 'Positive'
    elif score < -0.05: return 'Negative'
    else: return 'Neutral'

merged_df['sentiment_category'] = merged_df['sentiment_score'].apply(get_category)
category_returns = merged_df.groupby('sentiment_category')['daily_return'].mean().reset_index()

plt.figure(figsize=(7, 4))
sns.barplot(data=category_returns, x='sentiment_category', y='daily_return', order=['Negative', 'Neutral', 'Positive'], palette='coolwarm')
plt.title('Average Daily Return per Sentiment Category')
plt.ylabel('Mean Daily Return (%)')
plt.grid(axis='y')
plt.show()